# Task 1

## a) What is the fundamental difference between the model’s parameters and hyperparameters?

Parameters are values that the model learns automatically from the training data during the learning process. Examples of parameters in a neural network are weights and biases. During training, optimization algorithms such as Gradient Descent update these parameters to minimize the loss function and improve the model's predictions.

Hyperparameters are settings that are chosen before training begins. They control how the learning process is performed but are not learned from the data. Examples of hyperparameters include the learning rate, number of hidden layers, number of neurons per layer, batch size, and number of training epochs.

In summary, parameters are learned by the model during training, whereas hyperparameters are specified by the user before training starts.

## b) What is the validation set used for? What would happen if samples from the train/test set were used for the same purpose?

A validation set is used to evaluate the model during training, tune hyperparameters, detect overfitting, and estimate how well the model generalizes to unseen data. It provides an unbiased evaluation during model development without affecting the final test results.

If samples from the training set were used for validation, the evaluation would be overly optimistic because the model has already used those samples to learn and update its parameters. As a result, the validation performance would not accurately reflect the model's ability to generalize to new data.

If samples from the test set were used for validation, information from the test set would influence model selection and hyperparameter tuning. This leads to data leakage and makes the final test evaluation unreliable. Therefore, the test set should only be used after the model and all hyperparameters have been finalized.


# Task 2: Overfitting Analysis

## Scenario

An MLP (Multi-Layer Perceptron) achieves a very low error on the training data but produces a high error on the test data.

---

## What is this phenomenon called?

This phenomenon is called **Overfitting**.

Overfitting occurs when a machine learning model learns the training data too well, including its specific details, noise, and irregular patterns. Instead of learning general features that can be applied to new data, the model effectively memorizes the training examples.

As a result, the model performs very well on the training dataset but poorly on unseen test data because it cannot generalize its knowledge to new samples.

### Example

Suppose an MLP is trained to recognize handwritten digits from the MNIST dataset.

After training:

- Training Accuracy = 99.8%
- Training Error = 0.2%

However, when evaluated on unseen test images:

- Test Accuracy = 80%
- Test Error = 20%

Although the model performs almost perfectly on the training data, it fails to achieve similar performance on new data. This indicates that the model has overfitted the training dataset.

---

## What could cause this problem?

Several factors can lead to overfitting.

### 1. Model Complexity is Too High

If the neural network contains too many hidden layers or too many neurons, it becomes powerful enough to memorize the training data rather than learning general patterns.

#### Example

A problem that can be solved with:

- 2 hidden layers
- 128 neurons per layer

may overfit if a network with:

- 10 hidden layers
- 2000 neurons per layer

is used.

The larger network has significantly more parameters and may simply memorize the training examples.

---

### 2. Insufficient Training Data

When the training dataset is too small, the model has access to only a limited number of examples.

As a result, it may memorize those examples instead of learning the true underlying patterns.

#### Example

Training a digit recognition model using only 100 images is much more likely to cause overfitting than training it using 60,000 images.

---

### 3. Training for Too Many Epochs

If training continues for too long, the model may eventually start fitting noise and small variations in the training data.

Initially, both training and validation performance improve. However, after a certain point, validation performance begins to decrease while training performance continues to improve.

#### Example

| Epoch | Training Accuracy | Validation Accuracy |
| ----- | ----------------- | ------------------- |
| 5     | 90%               | 88%                 |
| 10    | 95%               | 93%                 |
| 20    | 99%               | 94%                 |
| 40    | 99.9%             | 89%                 |

The decrease in validation accuracy indicates overfitting.

---

### 4. Noisy or Incorrect Data

Datasets may contain:

- Incorrect labels
- Outliers
- Measurement errors

A highly flexible neural network may attempt to learn these incorrect patterns.

#### Example

If an image of the digit "3" is accidentally labeled as "8", the model may try to memorize this mistake, reducing its ability to generalize.

---

### 5. Lack of Regularization

Regularization techniques help prevent the model from becoming excessively specialized to the training data.

Without regularization, the model is free to develop very large and highly specific weights, increasing the risk of overfitting.

---

## What methods could you apply to address it?

Several techniques can be used to reduce overfitting and improve generalization.

### 1. Early Stopping

Early stopping monitors the validation loss during training.

Training is stopped when the validation performance no longer improves.

This prevents the model from memorizing the training data.

#### Example

If the validation loss stops improving after epoch 15, training can be terminated instead of continuing to epoch 50.

---

### 2. Dropout

Dropout randomly disables a percentage of neurons during training.

As a result, the network cannot rely too heavily on any particular neuron and is forced to learn more robust features.

#### Example

Using a dropout rate of 0.5 means that approximately 50% of neurons are randomly deactivated during each training step.

---

### 3. L1 and L2 Regularization

Regularization adds a penalty term to the loss function.

This discourages extremely large weight values and encourages simpler models.

Simpler models generally generalize better to unseen data.

---

### 4. Increase the Amount of Training Data

Providing more training examples helps the model learn the true data distribution rather than memorizing individual samples.

A larger dataset usually improves generalization performance.

---

### 5. Data Augmentation

Data augmentation artificially increases the size of the dataset by creating modified versions of existing samples.

Common augmentation techniques include:

- Rotation
- Scaling
- Translation
- Flipping
- Cropping

This exposes the model to more diverse examples.

---

### 6. Reduce Model Complexity

Using fewer layers or fewer neurons decreases the model's capacity to memorize the training data.

A simpler model often generalizes better.

#### Example

Instead of:

- 5 hidden layers
- 512 neurons per layer

use:

- 2 hidden layers
- 128 neurons per layer

if the task does not require a highly complex architecture.

---

## Conclusion

The phenomenon described in this scenario is called **overfitting**. It occurs when a model performs extremely well on training data but poorly on unseen test data because it has learned specific details and noise instead of general patterns.

Common causes include excessive model complexity, insufficient training data, training for too many epochs, noisy data, and lack of regularization. Overfitting can be reduced through techniques such as early stopping, dropout, regularization, data augmentation, increasing the dataset size, and simplifying the model architecture.

These methods improve the model's ability to generalize and achieve better performance on unseen data.


# MNIST Training


In [5]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd
import random

## Data Processing

The dataset is already loaded and split into a training, validation and testing set.


In [6]:
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# MNIST train has 60k images; we split 80/20
ds_train, ds_val, ds_test = tfds.load(
    "mnist",
    split=["train[:80%]", "train[80%:]", "test"],
    as_supervised=True,
    shuffle_files=True,
    batch_size=32
)

# Normalize datasets
normalize = lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.int32))
ds_train = ds_train.map(normalize)
ds_val   = ds_val.map(normalize)
ds_test  = ds_test.map(normalize)

# TODO: Your Code here
